In [ ]:
# ============================================================
# CELL 1: INSTALLS & IMPORTS
# ============================================================
!pip install anthropic requests pandas tqdm -q

import anthropic
import requests
import pandas as pd
import numpy as np
import json
import time
import re
from tqdm.notebook import tqdm
from google.colab import userdata

print("✓ All dependencies loaded.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 627.5/627.5 kB 7.6 MB/s eta 0:00:00
✓ All dependencies loaded.


In [ ]:
# ============================================================
# CELL 2: ANTHROPIC API KEY & CLIENT SETUP
# ============================================================
!pip install anthropic -q

import anthropic

ANTHROPIC_API_KEY = userdata.get('ANTHROPIC_API_KEY')
client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

# Validate the key works
test = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=50,
    messages=[{"role": "user", "content": "Reply with exactly: CONNECTION_OK"}]
)
response_text = test.content[0].text.strip()

if "CONNECTION_OK" in response_text:
    print("✓ Anthropic API key validated. Claude Sonnet connected.")
else:
    print(f"⚠ Unexpected response: {response_text}")

✓ Anthropic API key validated. Claude Sonnet connected.


In [ ]:
# ============================================================
# CELL 3: POLYMARKET ACTIVE EVENTS PULL
# ============================================================

BASE_URL = "https://gamma-api.polymarket.com/events"
LIMIT = 100  # max per page

def fetch_all_active_events():
    """
    Paginate through the Gamma API and pull all active, non-closed events.
    Returns a list of raw event dictionaries.
    """
    all_events = []
    offset = 0

    print("Fetching active Polymarket events...")

    while True:
        params = {
            "active": "true",
            "closed": "false",
            "limit": LIMIT,
            "offset": offset,
        }

        response = requests.get(BASE_URL, params=params, timeout=30)
        response.raise_for_status()
        batch = response.json()

        # If the response is empty, we've exhausted all pages
        if not batch:
            break

        all_events.extend(batch)
        print(f"  Pulled {len(batch)} events (total so far: {len(all_events)})")

        # If we got fewer than LIMIT, we're on the last page
        if len(batch) < LIMIT:
            break

        offset += LIMIT

        # Polite pause to avoid rate limiting
        time.sleep(0.3)

    print(f"\n✓ Total active events pulled: {len(all_events)}")
    return all_events


def parse_events_to_dataframe(events):
    """
    Extract the fields we need for classification from raw event JSON.
    Each row = one Polymarket event.
    """
    rows = []

    for event in events:
        # Extract tag labels from the array of tag objects
        tags_raw = event.get("tags", [])
        if tags_raw and isinstance(tags_raw, list):
            tag_labels = [t.get("label", "") for t in tags_raw if isinstance(t, dict)]
        else:
            tag_labels = []

        # Count sub-markets under this event
        sub_markets = event.get("markets", [])

        rows.append({
            "event_id":             event.get("id", ""),
            "title":                event.get("title", ""),
            "description":          event.get("description", ""),
            "polymarket_category":  event.get("category", ""),
            "tags":                 tag_labels,
            "tags_str":             " | ".join(tag_labels),  # flat string for display
            "num_sub_markets":      len(sub_markets) if sub_markets else 0,
            "volume":               event.get("volume", 0),
            "liquidity":            event.get("liquidity", 0),
            "start_date":           event.get("startDate", ""),
            "end_date":             event.get("endDate", ""),
        })

    df = pd.DataFrame(rows)
    return df


# ---- EXECUTE ----
raw_events = fetch_all_active_events()
df_events = parse_events_to_dataframe(raw_events)

# ---- QUICK DIAGNOSTIC ----
print(f"\nDataFrame shape: {df_events.shape}")
print(f"Events with descriptions: {df_events['description'].notna().sum()} / {len(df_events)}")
print(f"Events with tags: {(df_events['tags'].apply(len) > 0).sum()} / {len(df_events)}")
print(f"Events with no tags: {(df_events['tags'].apply(len) == 0).sum()}")
print(f"\nPolymarket category distribution:")
print(df_events['polymarket_category'].value_counts().to_string())
print(f"\nSample rows:")
df_events[['event_id', 'title', 'tags_str', 'polymarket_category']].head(10)

Fetching active Polymarket events...
  Pulled 100 events (total so far: 100)
  Pulled 100 events (total so far: 200)
  Pulled 100 events (total so far: 300)
  Pulled 100 events (total so far: 400)
  Pulled 100 events (total so far: 500)
  Pulled 100 events (total so far: 600)
  Pulled 100 events (total so far: 700)
  Pulled 100 events (total so far: 800)
  Pulled 100 events (total so far: 900)
  Pulled 100 events (total so far: 1000)
  Pulled 100 events (total so far: 1100)
  Pulled 100 events (total so far: 1200)
  Pulled 100 events (total so far: 1300)
  Pulled 100 events (total so far: 1400)
  Pulled 100 events (total so far: 1500)
  Pulled 100 events (total so far: 1600)
  Pulled 100 events (total so far: 1700)
  Pulled 100 events (total so far: 1800)
  Pulled 100 events (total so far: 1900)
  Pulled 100 events (total so far: 2000)
  Pulled 100 events (total so far: 2100)
  Pulled 100 events (total so far: 2200)
  Pulled 100 events (total so far: 2300)
  Pulled 100 events (total so

,event_id,title,tags_str,polymarket_category
0,16167,MicroStrategy sells any Bitcoin by ___ ?,Finance | Economy | Business | 2025 Prediction...,
1,16183,Kraken IPO by ___ ?,exchange | Tech | Crypto | Finance | Business ...,
2,16263,Macron out by...?,France | Politics | Macron | World | 2025 Pred...,
3,16423,UK election called by...?,Starmer | uk | pedophile | England,
4,17526,China x India military clash by...?,India | Politics | China | Geopolitics | World,
5,17549,NATO/EU troops fighting in Ukraine by...?,Russia | Politics | Geopolitics | Ukraine | Wo...,
6,17725,Starmer out by...?,Starmer | uk | keir | Grooming Gangs | Politic...,
7,17858,Ukraine recognizes Russian sovereignty over it...,World | Trump Presidency | putin | Trump | Pol...,
8,18558,Ukraine election called by...?,World | Geopolitics | Ukraine | Politics | For...,
9,18571,Will any country leave NATO by...?,World | NATO | Geopolitics | Politics,


In [ ]:
# ============================================================
# CELL 4: STATIC DICTIONARY ROUTER (PASS 1) — CORRECTED
# ============================================================

import re

# ---- PARSE TAGS INTO USABLE FORMAT ----
def parse_tags(t):
    """Handle tags whether they're already lists or string representations."""
    if isinstance(t, list):
        return t
    if isinstance(t, str) and t.strip():
        try:
            result = ast.literal_eval(t)
            return result if isinstance(result, list) else []
        except:
            return []
    return []

df_events['tags_parsed'] = df_events['tags'].apply(parse_tags)
df_events['tags_lower'] = df_events['tags_parsed'].apply(lambda x: [str(t).lower() for t in x])
df_events['title_lower'] = df_events['title'].str.lower()

# Quick sanity check
tags_populated = (df_events['tags_lower'].apply(len) > 0).sum()
print(f"Sanity check: {tags_populated} / {len(df_events)} events have tags\n")


# ---- REFERENCE SETS ----

SPORTS_TAGS = {
    'sports', 'soccer', 'hockey', 'basketball', 'cricket', 'esports',
    'games', 'football', 'tennis', 'baseball', 'golf', 'mma', 'boxing',
    'nfl', 'nba', 'mlb', 'nhl', 'ncaa', 'ufc', 'fifa', 'rugby',
    'afl', 'cycling', 'olympics', 'f1', 'formula 1', 'nascar',
    'premier league', 'champions league', 'la liga', 'serie a',
    'bundesliga', 'ligue 1',
}

ELECTION_PATTERNS = [
    r'election winner',
    r'house election',
    r'senate election',
    r'governor election',
    r'presidential election',
    r'parliamentary election',
    r'mayoral election',
    r'primary winner',
    r'caucus winner',
    r'next president of',
    r'next prime minister',
]

WEATHER_PATTERNS = [
    r'temperature in ',
    r'highest temperature',
    r'lowest temperature',
    r'precipitation in ',
    r'rainfall in ',
    r'snowfall in ',
    r'hurricane .* category',
    r'will .* earthquake',
]


# ---- STATIC CLASSIFICATION FUNCTION ----

def static_classify(row):
    """
    Attempt to classify an event using deterministic rules.
    Returns (category, confidence, method) or None if uncertain.

    Rules are applied in priority order. First match wins.
    Every rule requires multiple confirming signals to avoid false positives.
    """
    title = row['title_lower']
    tags = set(row['tags_lower'])

    # ── RULE 1: CRYPTO ──────────────────────────────────────────
    # 1a. "Up or Down" title + crypto-related tags (5-min price bets)
    if 'up or down' in title:
        if tags.intersection({'crypto prices', 'crypto', 'up or down'}):
            return ('Crypto', 0.99, 'static_crypto_updown')

    # 1b. "Crypto Prices" tag present (covers price targets, ATH, FDV,
    #     volatility indices, best candle, etc. — all verified as Crypto)
    if 'crypto prices' in tags:
        return ('Crypto', 0.95, 'static_crypto_prices_tag')

    # ── RULE 2: SPORTS ──────────────────────────────────────────
    # 2a. "Team vs Team" pattern + at least one sports tag
    if re.search(r'\bvs\.?\s', title) and tags.intersection(SPORTS_TAGS):
        return ('Sports', 0.99, 'static_sports_vs')

    # 2b. Sports-specific event keywords + sports tags
    sports_keywords = [
        'mvp', 'draft pick', 'world series', 'super bowl', 'grand prix',
        "ballon d'or", 'golden boot', 'cy young', 'heisman',
        'halftime result', 'full-time result', 'match result',
        'total goals', 'total points', 'total runs',
        'winner of leg', 'aggregate winner',
    ]
    if tags.intersection(SPORTS_TAGS) and any(kw in title for kw in sports_keywords):
        return ('Sports', 0.95, 'static_sports_keyword')

    # ── RULE 3: ELECTIONS ────────────────────────────────────────
    if any(re.search(p, title) for p in ELECTION_PATTERNS):
        return ('Elections', 0.97, 'static_elections_pattern')

    # ── RULE 4: CLIMATE AND WEATHER ──────────────────────────────
    if any(re.search(p, title) for p in WEATHER_PATTERNS):
        return ('Climate and Weather', 0.97, 'static_weather_pattern')

    # ── NO MATCH → ROUTE TO LLM ─────────────────────────────────
    return None


# ---- APPLY ----
df_events['static_result'] = df_events.apply(static_classify, axis=1)

# Split into statically classified vs needs-LLM
static_mask = df_events['static_result'].notna()
df_static = df_events[static_mask].copy()
df_needs_llm = df_events[~static_mask].copy()

# Unpack the (category, confidence, method) tuples
df_static['kalshi_category'] = df_static['static_result'].apply(lambda x: x[0])
df_static['confidence']      = df_static['static_result'].apply(lambda x: x[1])
df_static['method']          = df_static['static_result'].apply(lambda x: x[2])
df_static['rationale']       = 'Static dictionary match'


# ---- DIAGNOSTICS ----
print(f"STATIC ROUTER RESULTS")
print(f"{'='*55}")
print(f"  Total events:            {len(df_events):,}")
print(f"  Statically classified:   {len(df_static):,}  ({len(df_static)/len(df_events)*100:.1f}%)")
print(f"  Routed to LLM:           {len(df_needs_llm):,}  ({len(df_needs_llm)/len(df_events)*100:.1f}%)")
print(f"\n  Category breakdown (static only):")
for cat, count in df_static['kalshi_category'].value_counts().items():
    print(f"    {cat:<25} {count:>5}")
print(f"\n  Method breakdown:")
for method, count in df_static['method'].value_counts().items():
    print(f"    {method:<30} {count:>5}")

# ---- SPOT-CHECK: Show 5 random examples per category ----
print(f"\n{'='*55}")
print(f"SPOT-CHECK: 5 random examples per static category")
print(f"{'='*55}")
for cat in df_static['kalshi_category'].unique():
    subset = df_static[df_static['kalshi_category'] == cat]
    sample = subset.sample(min(5, len(subset)), random_state=42)
    print(f"\n  ── {cat} ──")
    for _, row in sample.iterrows():
        print(f"    • {row['title']}")

# ---- SPOT-CHECK: Show 10 random LLM-bound events ----
print(f"\n{'='*55}")
print(f"SPOT-CHECK: 10 random events routed to LLM")
print(f"{'='*55}")
for _, row in df_needs_llm.sample(10, random_state=42).iterrows():
    tags_preview = row['tags_parsed'][:4]
    print(f"  • {row['title']}")
    print(f"    Tags: {tags_preview}")

Sanity check: 10731 / 10731 events have tags

STATIC ROUTER RESULTS
  Total events:            10,731
  Statically classified:   8,651  (80.6%)
  Routed to LLM:           2,080  (19.4%)

  Category breakdown (static only):
    Sports                     4358
    Crypto                     3388
    Elections                   768
    Climate and Weather         137

  Method breakdown:
    static_sports_vs                4346
    static_crypto_updown            3277
    static_elections_pattern         768
    static_weather_pattern           137
    static_crypto_prices_tag         111
    static_sports_keyword             12

SPOT-CHECK: 5 random examples per static category

  ── Crypto ──
    • Solana Up or Down - April 13, 11:20AM-11:25AM ET
    • Bitcoin Up or Down - April 13, 10:35AM-10:40AM ET
    • Hyperliquid Up or Down - April 13, 4:30AM-4:35AM ET
    • Bitcoin Up or Down - April 13, 6PM ET
    • Dogecoin Up or Down - April 12, 11:35AM-11:40AM ET

  ── Elections ──
    • VT-A

In [ ]:
# ============================================================
# CELL 5 (REVISED): LLM CLASSIFIER (PASS 2) — BATCHED
# ============================================================
# Changes from original:
#   - Batches 10 events per API call (was 1:1)
#   - ~180 calls instead of ~1,800 → ~5-10 min, ~$8-10
#   - Returns JSON array like Cell 9
#   - Same system prompt, same category validation
#
# Estimated: ~180 API calls · ~$8-10 · ~5-10 min runtime
# ============================================================

import json
import time
import re
from tqdm.notebook import tqdm

MODEL = "claude-sonnet-4-20250514"
BATCH_SIZE = 10

# ---- VALID CATEGORIES ----
VALID_CATEGORIES = {
    'Climate and Weather', 'Companies', 'Crypto', 'Economics',
    'Elections', 'Entertainment', 'Financials', 'Mentions',
    'Politics', 'Science and Technology', 'Social', 'Sports', 'World'
}

# ---- SYSTEM PROMPT: KALSHI BOUNDARY ANALYSIS ----

SYSTEM_PROMPT = """You are a prediction market classification engine. You will be given a batch of Polymarket events. For EACH event, assign it to exactly ONE of the 13 Kalshi categories listed below.

Return ONLY a valid JSON array — no other text, no markdown, no explanation outside the JSON:
[{"event": 1, "category": "...", "confidence": 0.XX, "rationale": "..."}, {"event": 2, ...}, ...]

THE 13 KALSHI CATEGORIES AND THEIR BOUNDARY RULES:

1. CLIMATE AND WEATHER
Definition: Meteorological events, geological phenomena, and macro-environmental trends. Specific weather occurrences, natural disasters, and governmental climate goals or energy consumption benchmarks.
Includes: Government climate goal achievements (resolution depends on emissions data, not legislation). Global primary energy consumption by source (environmental transition focus, not financial performance).
Excludes: Global CO2 atmospheric concentration levels → World. Building nuclear-powered data centers → Science and Technology.
Decision rules:
- Climate and Weather vs Economics: Source of primary energy consumption → Climate; price of energy commodities → Economics.
- Climate and Weather vs World: Specific disaster/weather event/national climate goal → Climate; extreme long-term planetary existential metric → World.

2. COMPANIES
Definition: Corporate actions, leadership changes, and product milestones of specific named businesses. Internal corporate governance, CEO successions, major antitrust lawsuits involving specific companies, and technological announcements tied to corporate entities.
Includes: DOJ antitrust lawsuit against a specific company (resolution hinges on corporate outcome, not broad policy). Company announcing AGI (corporate PR claim, not universally verified scientific milestone).
Excludes: Earnings call word mentions → Mentions. Companies racing to IPO first → Financials.
Decision rules:
- Companies vs Financials: Corporate leadership/lawsuits/product announcements → Companies; IPO timing or stock performance → Financials.
- Companies vs Science and Technology: Which company will announce a tech breakthrough → Companies; when will a general tech breakthrough be achieved by humanity → Science and Technology.

3. CRYPTO
Definition: Pricing, market capitalization, and technical deployment milestones of cryptocurrencies and blockchain networks. Price targets for specific tokens, fully diluted valuation (FDV) metrics, and mainnet launches.
Includes: Mainnet deployments (blockchain-specific infrastructure, not general software). FDV targets (crypto-native market cap metrics, not traditional equities).
Excludes: Crypto founders reaching trillionaire status → Economics.
Decision rules:
- Crypto vs Financials: Blockchain token/network/crypto FDV → Crypto; fiat currency pair or traditional commodity → Financials.

4. ECONOMICS
Definition: Macro-level economic indicators and wealth milestones. National labor statistics, broad macroeconomic health indicators, and net worth achievements of global billionaires.
Includes: Individuals becoming world's first trillionaire (macro-economic wealth milestone). Maximum unemployment rates (pure statistical economic output).
Excludes: Specific commodity prices → Financials. Global primary energy consumption by source → Climate and Weather.
Decision rules:
- Economics vs Financials: Macroeconomic statistics or general wealth milestones → Economics; daily trading prices of financial instruments → Financials.

5. ELECTIONS
Definition: Outcomes of leadership selection processes, both democratic and non-democratic. Who will become the next head of state, party leader, or supreme religious figure.
Includes: Next Pope (fundamentally a leadership election process). Next leader of the Chinese Communist Party (predicting specific leadership succession).
Excludes: World leaders leaving office early → Politics. Presidential Medal of Freedom recipients → World.
Decision rules:
- Elections vs Politics: Predicting who will win or succeed to a leadership role → Elections; predicting if an existing leader will leave office early or a government will collapse → Politics.

6. ENTERTAINMENT
Definition: Outcomes related to pop culture, media production, and the arts. Awards ceremonies, talent attachments to major franchises, box office or critical performance, music charting.
Includes: Performing the next James Bond song (formal professional attachment within film/music industry).
Excludes: Celebrity weddings → Social. Corporate antitrust lawsuits against entertainment companies → Companies.
Decision rules:
- Entertainment vs Social: Professional media release/casting/award → Entertainment; personal life event of a celebrity → Social.

7. FINANCIALS
Definition: Trading prices, indices, and financial market performances of traditional assets. Commodities, stock indices, fiat currency pairs, and comparative market races.
Includes: Race to IPO first between companies (financial market race). Maximum WTI front month settle price (exact futures contract pricing).
Excludes: Trillionaire net worth milestones → Economics. Earnings call mentions → Mentions.
Decision rules:
- Financials vs Companies: Comparing financial timelines of multiple entities → Financials; internal corporate governance or legal outcomes → Companies.
- Financials vs Economics: Exact trading prices → Financials; macro wealth and labor statistics → Economics.

8. MENTIONS
Definition: Predicting the literal words, phrases, or topics spoken by individuals during specific public events or broadcasts. Word-matching games based on transcripts.
Includes: Topics mentioned during earnings calls (resolution depends entirely on audio transcript).
Excludes: A company announcing AGI → Companies (fundamental shift in business, not transcript word-matching).
Decision rules:
- Mentions vs Companies: What a company will literally say on a specific call → Mentions; whether a company will actually do something → Companies.

9. POLITICS
Definition: Actions, stability, and legislative behaviors of governments and political figures outside of formal elections. Leaders leaving office early, judicial decisions, legislative control, international geopolitical stability.
Includes: World leaders leaving office early (government stability and term termination). Geopolitical conflicts, wars, sanctions, trade disputes, government shutdowns, diplomatic negotiations.
Excludes: Elections of new leaders → Elections. Presidential Medal of Freedom recipients → World. Politicians visiting specific states → World.
Decision rules:
- Politics vs Elections: Early exits/government collapses/policy changes → Politics; formal succession and voting outcomes → Elections.
- Politics vs World: Concrete geopolitical stability events → Politics; ceremonial government awards or physical state visits → World.

10. SCIENCE AND TECHNOLOGY
Definition: Major scientific milestones, space exploration, technological breakthroughs, and legal/infrastructure frameworks surrounding new technology. Nuclear fusion, Mars landings, AI copyright disputes.
Includes: SpaceX landing on Moon/Mars (milestone in human aerospace achievement, not standard corporate product launch). AI copyright lawsuits (fundamental legal precedents for AI as a technology).
Excludes: Corporate announcements of AGI → Companies. Primary energy consumption by source → Climate and Weather.
Decision rules:
- Science and Technology vs Companies: Landmark human achievement or legal framework for tech → Science and Technology; company announcing a product or internal milestone → Companies.

11. SOCIAL
Definition: Demographic shifts, cultural phenomena, and personal lives of public figures. State population changes, celebrity gossip, wedding parties.
Includes: State population decreases (demographic/sociological metric). Celebrity weddings and personal relationships.
Excludes: Entertainers performing a theme song → Entertainment. Athletes competing in sports → Sports.
Decision rules:
- Social vs Entertainment/Sports: Personal or private milestone for a celebrity/athlete → Social; their professional work → Entertainment or Sports.
- Social vs Economics: Demographic/population changes → Social; labor/employment/wealth metrics → Economics.

12. SPORTS
Definition: Outcomes, business operations, and player statistics of professional and international athletic competitions. Championships, franchise relocations, ownership changes, player performance.
Includes: Franchise relocations (sports franchises are athletic entities, not standard corporate businesses). Player becoming majority team owner (governance of sports leagues).
Excludes: Athletes in celebrity weddings → Social. Earnings calls for sports apparel companies → Mentions.
Decision rules:
- Sports vs Companies: Business of sports (ownership, relocation, expansion) → Sports, not Companies.
- Sports vs Social: On-field performance and league business → Sports; personal relationships of athletes → Social.

13. WORLD
Definition: Catch-all for global milestones, long-term existential planetary trends, and specific ceremonial or miscellaneous events that don't fit other buckets. Ultra-long-term predictions, extreme climate metrics, geopolitical expansion, executive ceremonial awards.
Includes: Atmospheric CO2 concentration levels (extreme long-term planetary-scale metric). Presidential Medal of Freedom recipients (ceremonial honor). Politicians' state visits (physical travel/logistics).
Excludes: Next Pope → Elections. General human achievements in space → Science and Technology.
Decision rules:
- World vs Politics: Ceremonial actions and physical travel → World; actual geopolitical stability → Politics.
- World vs Climate and Weather: Standard climate goals and natural disasters → Climate and Weather; broad existential metrics like total atmospheric CO2 → World.

IMPORTANT CLASSIFICATION PRINCIPLES:
- Kalshi categorizes by ACTION, not by ENTITY. "Elon Musk" could be Companies, Economics, Science and Technology, or World depending on what the market is asking.
- When in doubt between two categories, apply the specific decision rule for that pair.
- If the event truly does not fit ANY category, classify it as "World" (the catch-all) with low confidence.
- Always use the MOST SPECIFIC category available. Only use "World" if no other category fits.
- Return ONLY the JSON array. No other text."""


# ---- DESCRIPTION TRUNCATION ----
DESC_MAX_CHARS = 1500


def build_batch_prompt(batch_rows):
    """Build a classification prompt for a batch of events."""
    lines = []
    for i, (_, row) in enumerate(batch_rows.iterrows(), 1):
        title = row['title']
        description = str(row['description'])[:DESC_MAX_CHARS] if pd.notna(row['description']) else ''
        tags = row['tags_parsed'] if isinstance(row['tags_parsed'], list) else []
        tags_str = ', '.join(str(t) for t in tags) if tags else '(none)'

        lines.append(f"""EVENT {i}:
TITLE: {title}
DESCRIPTION: {description}
TAGS: {tags_str}""")

    return "\n\n".join(lines) + "\n\nClassify each event. Return ONLY the JSON array."


# ---- JSON EXTRACTION ----

def extract_json_array(text):
    """Extract a JSON array from model output."""
    text = text.strip()

    # Direct parse
    try:
        result = json.loads(text)
        if isinstance(result, list):
            return result
    except json.JSONDecodeError:
        pass

    # Strip markdown
    if '```' in text:
        cleaned = re.sub(r'```(?:json)?\s*', '', text)
        cleaned = re.sub(r'```', '', cleaned).strip()
        try:
            result = json.loads(cleaned)
            if isinstance(result, list):
                return result
        except json.JSONDecodeError:
            pass

    # Find [ ... ] block
    match = re.search(r'\[.*\]', text, re.DOTALL)
    if match:
        try:
            result = json.loads(match.group())
            if isinstance(result, list):
                return result
        except json.JSONDecodeError:
            pass

    return None


# ---- CATEGORY VALIDATION ----

def normalize_category(category_str):
    """Match model output to valid category names (case-insensitive)."""
    for vc in VALID_CATEGORIES:
        if vc.lower() == category_str.lower().strip():
            return vc
    return None


# ---- BATCH CLASSIFICATION ----

def classify_batch(batch_rows, max_retries=3):
    """Send a batch of events to Claude for classification."""
    prompt = build_batch_prompt(batch_rows)

    for attempt in range(max_retries):
        try:
            response = client.messages.create(
                model=MODEL,
                max_tokens=2000,
                system=SYSTEM_PROMPT,
                messages=[{"role": "user", "content": prompt}]
            )

            raw_text = response.content[0].text.strip()
            results = extract_json_array(raw_text)

            if results is None or len(results) == 0:
                if attempt < max_retries - 1:
                    time.sleep(2)
                    continue
                return [
                    {'category': 'PARSE_ERROR', 'confidence': 0.0,
                     'rationale': f'Could not extract JSON after {max_retries} attempts',
                     'error': True}
                    for _ in range(len(batch_rows))
                ]

            # Validate and normalize each result
            validated = []
            for i in range(len(batch_rows)):
                if i < len(results):
                    r = results[i]
                    raw_category = r.get('category', '')
                    category = normalize_category(raw_category)

                    if category is None:
                        validated.append({
                            'category': 'INVALID',
                            'confidence': 0.0,
                            'rationale': f'Unrecognized category: {raw_category}',
                            'error': True,
                        })
                    else:
                        validated.append({
                            'category': category,
                            'confidence': float(r.get('confidence', 0.0)),
                            'rationale': str(r.get('rationale', '')),
                            'error': False,
                        })
                else:
                    validated.append({
                        'category': 'MISSING',
                        'confidence': 0.0,
                        'rationale': 'Missing from batch response',
                        'error': True,
                    })

            return validated

        except Exception as e:
            if 'rate' in str(e).lower() or '429' in str(e):
                wait_time = min(2 ** (attempt + 2), 60)
                print(f"\n  ⏳ Rate limited. Waiting {wait_time}s...")
                time.sleep(wait_time)
                continue

            if attempt < max_retries - 1:
                time.sleep(2 ** attempt)
                continue

            return [
                {'category': 'API_ERROR', 'confidence': 0.0,
                 'rationale': f'API error: {str(e)[:150]}', 'error': True}
                for _ in range(len(batch_rows))
            ]


def classify_all_llm_events(df_llm, save_every=50):
    """Classify all LLM-bound events in batches with progress tracking."""
    results = []
    total = len(df_llm)
    total_batches = (total + BATCH_SIZE - 1) // BATCH_SIZE
    errors = 0

    print(f"Starting BATCHED LLM classification of {total:,} events via Claude Sonnet...")
    print(f"Batch size: {BATCH_SIZE} | Total batches: {total_batches}")
    print(f"Estimated time: {total_batches * 1.0 / 60:.0f}-{total_batches * 2.0 / 60:.0f} minutes")
    print(f"Estimated cost: ~${total_batches * 0.05:.2f}\n")

    for batch_idx in tqdm(range(total_batches), desc="Classifying"):
        start = batch_idx * BATCH_SIZE
        end = min(start + BATCH_SIZE, total)
        batch_rows = df_llm.iloc[start:end]

        batch_results = classify_batch(batch_rows)

        # Map results back to rows
        for i, (_, row) in enumerate(batch_rows.iterrows()):
            r = batch_results[i] if i < len(batch_results) else {
                'category': 'MISSING', 'confidence': 0.0,
                'rationale': 'Missing from response', 'error': True
            }
            r['event_id'] = row['event_id']
            r['title'] = row['title']
            results.append(r)

            if r.get('error', False):
                errors += 1

        # Pacing
        time.sleep(0.3)

        # Checkpoint
        if (batch_idx + 1) % save_every == 0:
            temp_df = pd.DataFrame(results)
            temp_df.to_csv('llm_classification_checkpoint.csv', index=False)
            pct = (batch_idx + 1) / total_batches * 100
            print(f"\n  💾 Checkpoint: batch {batch_idx+1}/{total_batches} ({pct:.1f}%) | {errors} errors")

    results_df = pd.DataFrame(results)
    results_df.to_csv('llm_classification_checkpoint.csv', index=False)
    print(f"\n✓ Classification complete: {total} events in {total_batches} batches, {errors} errors")
    return results_df


# ---- EXECUTE ----
df_llm_results = classify_all_llm_events(df_needs_llm)


# ---- POST-RUN DIAGNOSTICS ----
print(f"\n{'='*55}")
print(f"LLM CLASSIFICATION RESULTS")
print(f"{'='*55}")

error_count = df_llm_results['error'].sum()
print(f"  Total classified:  {len(df_llm_results):,}")
print(f"  Successful:        {len(df_llm_results) - error_count:,}")
print(f"  Errors:            {error_count}")

if error_count > 0:
    print(f"\n  Error breakdown:")
    for cat, count in df_llm_results[df_llm_results['error']]['category'].value_counts().items():
        print(f"    {cat}: {count}")

successful = df_llm_results[~df_llm_results['error']]

print(f"\n  Category distribution (LLM-classified):")
for cat, count in successful['category'].value_counts().items():
    pct = count / len(successful) * 100
    print(f"    {cat:<25} {count:>5}  ({pct:.1f}%)")

print(f"\n  Confidence distribution:")
print(f"    High (≥0.90):     {(successful['confidence'] >= 0.90).sum()}")
print(f"    Medium (0.7-0.9): {((successful['confidence'] >= 0.70) & (successful['confidence'] < 0.90)).sum()}")
print(f"    Low (<0.70):      {(successful['confidence'] < 0.70).sum()}")

low_conf = successful[successful['confidence'] < 0.70].head(5)
if len(low_conf) > 0:
    print(f"\n  Low-confidence examples:")
    for _, r in low_conf.iterrows():
        print(f"    • [{r['category']}] (conf: {r['confidence']}) {r['title']}")
        print(f"      Rationale: {r['rationale']}")

print(f"\n  High-confidence spot check (5 random):")
high_conf = successful[successful['confidence'] >= 0.90]
if len(high_conf) > 0:
    for _, r in high_conf.sample(min(5, len(high_conf)), random_state=42).iterrows():
        print(f"    • [{r['category']}] (conf: {r['confidence']}) {r['title']}")
        print(f"      Rationale: {r['rationale']}")


Starting BATCHED LLM classification of 2,080 events via Claude Sonnet...
Batch size: 10 | Total batches: 208
Estimated time: 3-7 minutes
Estimated cost: ~$10.40



Classifying:   0%|          | 0/208 [00:00<?, ?it/s]


  💾 Checkpoint: batch 50/208 (24.0%) | 0 errors

  💾 Checkpoint: batch 100/208 (48.1%) | 0 errors

  💾 Checkpoint: batch 150/208 (72.1%) | 0 errors

  💾 Checkpoint: batch 200/208 (96.2%) | 0 errors

✓ Classification complete: 2080 events in 208 batches, 0 errors

LLM CLASSIFICATION RESULTS
  Total classified:  2,080
  Successful:        2,080
  Errors:            0

  Category distribution (LLM-classified):
    Politics                    570  (27.4%)
    Sports                      451  (21.7%)
    Financials                  240  (11.5%)
    Crypto                      171  (8.2%)
    Entertainment               141  (6.8%)
    Companies                   123  (5.9%)
    Economics                    93  (4.5%)
    Elections                    83  (4.0%)
    Social                       66  (3.2%)
    Science and Technology       41  (2.0%)
    World                        38  (1.8%)
    Mentions                     33  (1.6%)
    Climate and Weather          30  (1.4%)

  Confidence

In [ ]:
# ============================================================
# CELL 6: MERGE, VALIDATE & EXPORT
# ============================================================

# ---- MERGE STATIC + LLM RESULTS ----

# Prepare LLM results to match static results format
df_llm_final = df_llm_results[~df_llm_results['error']].copy()
df_llm_final['method'] = 'llm_claude_sonnet'

# Prepare static results
df_static_final = df_static[['event_id', 'title', 'kalshi_category',
                              'confidence', 'method', 'rationale']].copy()

# Prepare LLM results
df_llm_merge = df_llm_final[['event_id', 'title', 'category',
                              'confidence', 'rationale', 'method']].copy()
df_llm_merge = df_llm_merge.rename(columns={'category': 'kalshi_category'})

# Combine
df_classified = pd.concat([df_static_final, df_llm_merge], ignore_index=True)

# ---- REJOIN WITH ORIGINAL EVENT DATA ----

# Merge back with the full event dataframe to get descriptions, tags, etc.
df_final = df_events.merge(
    df_classified[['event_id', 'kalshi_category', 'confidence', 'method', 'rationale']],
    on='event_id',
    how='left'
)

# ---- VALIDATION ----

unclassified = df_final['kalshi_category'].isna().sum()
total = len(df_final)
classified = total - unclassified

print(f"FINAL CLASSIFICATION REPORT")
print(f"{'='*55}")
print(f"  Total events:       {total:,}")
print(f"  Classified:         {classified:,}  ({classified/total*100:.1f}%)")
print(f"  Unclassified:       {unclassified}")

print(f"\n  Classification method breakdown:")
for method, count in df_final['method'].value_counts().items():
    pct = count / total * 100
    print(f"    {method:<30} {count:>5}  ({pct:.1f}%)")

print(f"\n  FULL CATEGORY DISTRIBUTION (Static + LLM combined):")
print(f"  {'-'*50}")
for cat, count in df_final['kalshi_category'].value_counts().items():
    pct = count / total * 100
    bar = '█' * int(pct)
    print(f"    {cat:<25} {count:>5}  ({pct:>5.1f}%) {bar}")

print(f"\n  Confidence summary:")
print(f"    Mean:    {df_final['confidence'].mean():.3f}")
print(f"    Median:  {df_final['confidence'].median():.3f}")
print(f"    Min:     {df_final['confidence'].min():.3f}")
print(f"    <0.70:   {(df_final['confidence'] < 0.70).sum()} events")
print(f"    <0.80:   {(df_final['confidence'] < 0.80).sum()} events")

# ---- EXPORT ----

# Select and order columns for the final output
export_cols = [
    'event_id', 'title', 'description', 'tags_str',
    'kalshi_category', 'confidence', 'method', 'rationale',
    'volume', 'liquidity', 'start_date', 'end_date', 'num_sub_markets'
]
df_export = df_final[export_cols].sort_values('kalshi_category')
df_export.to_csv('polymarket_classified_final.csv', index=False)

print(f"\n{'='*55}")
print(f"✓ Exported to 'polymarket_classified_final.csv'")
print(f"  Shape: {df_export.shape}")
print(f"  Columns: {list(df_export.columns)}")

# ---- SPOT CHECK: 3 random events per category ----
print(f"\n{'='*55}")
print(f"SPOT CHECK: 3 random events per category")
print(f"{'='*55}")
for cat in sorted(df_export['kalshi_category'].unique()):
    subset = df_export[df_export['kalshi_category'] == cat]
    sample = subset.sample(min(3, len(subset)), random_state=42)
    print(f"\n  ── {cat} ({len(subset)} events) ──")
    for _, row in sample.iterrows():
        conf = row['confidence']
        method = 'S' if 'static' in str(row['method']) else 'L'
        print(f"    [{method}] (conf: {conf}) {row['title']}")

FINAL CLASSIFICATION REPORT
  Total events:       10,913
  Classified:         10,913  (100.0%)
  Unclassified:       0

  Classification method breakdown:
    static_sports_vs                4364  (40.0%)
    static_crypto_updown            3435  (31.5%)
    llm_claude_sonnet               2086  (19.1%)
    static_elections_pattern         768  (7.0%)
    static_weather_pattern           137  (1.3%)
    static_crypto_prices_tag         111  (1.0%)
    static_sports_keyword             12  (0.1%)

  FULL CATEGORY DISTRIBUTION (Static + LLM combined):
  --------------------------------------------------
    Sports                     4829  ( 44.2%) ████████████████████████████████████████████
    Crypto                     3717  ( 34.1%) ██████████████████████████████████
    Elections                   851  (  7.8%) ███████
    Politics                    570  (  5.2%) █████
    Financials                  242  (  2.2%) ██
    Climate and Weather         167  (  1.5%) █
    Entertainme

In [ ]:
# ============================================================
# CELL 6B: POST-CLASSIFICATION CONSISTENCY ENFORCER
# ============================================================
# Catches and corrects two classes of error:
#
# 1. STATIC ROUTER LEAKS — events that matched a static rule
#    but belong in a different category (e.g., stock "Up or Down"
#    markets caught by the crypto router).
#
# 2. LLM INCONSISTENCIES — structurally identical markets that
#    the LLM classified differently because it focused on entity
#    instead of action (e.g., "Trump # posts" → Mentions but
#    "Musk # tweets" → Social).
#
# To add a new rule: append a dict to CONSISTENCY_RULES below.
# Each rule has:
#   - name:       human-readable label for logging
#   - match:      function(row) → True if this row should be checked
#   - correct_to: the category it should be
#   - confidence: confidence to assign
#   - rationale:  explanation string
# ============================================================

CONSISTENCY_RULES = [

    # ── RULE 1: Stock/Commodity/Index "Up or Down" → Financials ──
    # These have "Up or Down" in the title but are traditional
    # financial assets, not crypto. Detected by having Finance/
    # Stocks/Equities/Commodities tags WITHOUT Crypto tags.
    {
        'name': 'financial_updown_leak',
        'match': lambda row: (
            'up or down' in str(row.get('title', '')).lower()
            and _has_any_tag(row, {'finance', 'stocks', 'equities',
                                   'commodities', 'indicies',
                                   'finance updown', 'equity daily pyth',
                                   'daily-close'})
            and not _has_any_tag(row, {'crypto', 'crypto prices'})
        ),
        'correct_to': 'Financials',
        'confidence': 0.99,
        'rationale': 'Consistency rule: stock/commodity/index Up or Down market',
    },

    # ── RULE 2: Post/Tweet counting markets → Mentions ──
    # Any market that asks "# posts", "# tweets", or "# tweet"
    # for any person is a content-counting market, regardless of
    # who the person is. Structurally identical to transcript
    # word-matching (Mentions).
    {
        'name': 'post_tweet_counting',
        'match': lambda row: (
            bool(re.search(
                r'#\s*(posts?|tweets?)\b',
                str(row.get('title', '')),
                re.IGNORECASE
            ))
        ),
        'correct_to': 'Mentions',
        'confidence': 0.85,
        'rationale': 'Consistency rule: post/tweet counting market → Mentions',
    },

    # ── RULE 3: "What will [person] say" markets → Mentions ──
    # Any market predicting literal words spoken by someone.
    {
        'name': 'what_will_say',
        'match': lambda row: (
            bool(re.search(
                r'what will .+ (say|post)\b',
                str(row.get('title', '')),
                re.IGNORECASE
            ))
        ),
        'correct_to': 'Mentions',
        'confidence': 0.95,
        'rationale': 'Consistency rule: predicting literal words → Mentions',
    },

    # ── RULE 4: "What will be said during" markets → Mentions ──
    {
        'name': 'what_will_be_said',
        'match': lambda row: (
            bool(re.search(
                r'what will be said',
                str(row.get('title', '')),
                re.IGNORECASE
            ))
        ),
        'correct_to': 'Mentions',
        'confidence': 0.95,
        'rationale': 'Consistency rule: predicting literal words during event → Mentions',
    },

    # ────────────────────────────────────────────────────────────
    # ADD NEW RULES HERE. Format:
    #
    # {
    #     'name': 'descriptive_name',
    #     'match': lambda row: <boolean condition>,
    #     'correct_to': '<Kalshi category>',
    #     'confidence': <float>,
    #     'rationale': '<explanation>',
    # },
    # ────────────────────────────────────────────────────────────
]


# ---- HELPER FUNCTION ----

def _has_any_tag(row, target_tags):
    """Check if a row's tags_str contains any of the target tags."""
    tags_str = str(row.get('tags_str', ''))
    if not tags_str or tags_str == 'nan':
        return False
    row_tags = {t.strip().lower() for t in tags_str.split('|')}
    return bool(row_tags.intersection(target_tags))


# ---- APPLY ALL RULES ----

import re

total_patches = 0
patch_log = []

for rule in CONSISTENCY_RULES:
    rule_name = rule['name']
    matches = df_export.apply(rule['match'], axis=1)

    # Only patch events that are currently in a DIFFERENT category
    already_correct = df_export['kalshi_category'] == rule['correct_to']
    needs_patch = matches & ~already_correct

    count = needs_patch.sum()
    if count > 0:
        # Log what we're changing
        for _, r in df_export[needs_patch].iterrows():
            patch_log.append({
                'rule': rule_name,
                'event_id': r['event_id'],
                'title': r['title'],
                'old_category': r['kalshi_category'],
                'new_category': rule['correct_to'],
            })

        # Apply the patch
        df_export.loc[needs_patch, 'kalshi_category'] = rule['correct_to']
        df_export.loc[needs_patch, 'confidence'] = rule['confidence']
        df_export.loc[needs_patch, 'method'] = f'consistency_{rule_name}'
        df_export.loc[needs_patch, 'rationale'] = rule['rationale']

        total_patches += count
        print(f"  ✓ {rule_name}: patched {count} events → {rule['correct_to']}")
    else:
        print(f"  – {rule_name}: no corrections needed")


# ---- PATCH LOG ----

if patch_log:
    print(f"\nDETAILED PATCH LOG ({total_patches} total corrections):")
    print(f"{'-'*70}")
    for p in patch_log:
        print(f"  [{p['old_category']} → {p['new_category']}] {p['title']}")

    df_patch_log = pd.DataFrame(patch_log)
    df_patch_log.to_csv('consistency_patch_log.csv', index=False)
    print(f"\n  Patch log saved to consistency_patch_log.csv")


# ---- RE-EXPORT ----

df_export.to_csv('polymarket_classified_final.csv', index=False)

print(f"\nFINAL CATEGORY DISTRIBUTION:")
print(f"{'-'*55}")
total = len(df_export)
for cat, count in df_export['kalshi_category'].value_counts().items():
    pct = count / total * 100
    bar = '█' * int(pct)
    print(f"  {cat:<25} {count:>5}  ({pct:>5.1f}%) {bar}")

print(f"\n✓ {total_patches} total corrections applied.")
print(f"✓ Final CSV exported to polymarket_classified_final.csv")

  ✓ financial_updown_leak: patched 70 events → Financials
  ✓ post_tweet_counting: patched 9 events → Mentions
  ✓ what_will_say: patched 1 events → Mentions
  – what_will_be_said: no corrections needed

DETAILED PATCH LOG (80 total corrections):
----------------------------------------------------------------------
  [Crypto → Financials] Natural Gas (NG) Up or Down on April 9?
  [Crypto → Financials] Silver (XAGUSD) Up or Down on April 9?
  [Crypto → Financials] Gold (XAUUSD) Up or Down on April 9?
  [Crypto → Financials] EWY (EWY) Up or Down on April 9?
  [Crypto → Financials] SPY (SPY) Up or Down on April 9?
  [Crypto → Financials] QQQ (QQQ) Up or Down on April 9?
  [Crypto → Financials] Airbnb (ABNB) Up or Down on April 9?
  [Crypto → Financials] Opendoor (OPEN) Up or Down on April 9?
  [Crypto → Financials] Apple (AAPL) Up or Down on April 10?
  [Crypto → Financials] Natural Gas (NG) Up or Down on April 10?
  [Crypto → Financials] WTI Crude Oil (WTI) Up or Down on April 10?
  [Cr

In [ ]:
# ============================================================
# CELL 7: PULL ALL ACTIVE KALSHI EVENTS
# ============================================================
# No authentication required for reading market data.
# Uses cursor-based pagination with 200 events per page.
# ============================================================

KALSHI_BASE = "https://api.elections.kalshi.com/trade-api/v2"

def fetch_all_kalshi_events():
    """Paginate through Kalshi's events endpoint and pull all active events."""
    all_events = []
    cursor = None

    print("Fetching active Kalshi events...")

    while True:
        params = {
            "status": "open",
            "with_nested_markets": "true",
            "limit": 200,
        }
        if cursor:
            params["cursor"] = cursor

        response = requests.get(f"{KALSHI_BASE}/events", params=params, timeout=30)
        response.raise_for_status()
        data = response.json()

        batch = data.get("events", [])
        if not batch:
            break

        all_events.extend(batch)
        print(f"  Pulled {len(batch)} events (total so far: {len(all_events)})")

        cursor = data.get("cursor")
        if not cursor:
            break

        time.sleep(0.3)

    print(f"\n✓ Total active Kalshi events pulled: {len(all_events)}")
    return all_events


def parse_kalshi_events(events):
    """Extract fields needed for matching from raw Kalshi event JSON."""
    rows = []

    for event in events:
        # Collect nested market titles and tickers
        nested_markets = event.get("markets", [])
        market_titles = []
        market_tickers = []
        market_rules = []

        for m in nested_markets:
            if m.get("status") in ("open", "unopened"):
                market_titles.append(m.get("title", ""))
                market_tickers.append(m.get("ticker", ""))
                rules = m.get("rules_primary", "")
                if rules:
                    market_rules.append(rules)

        rows.append({
            "kalshi_event_ticker":  event.get("event_ticker", ""),
            "kalshi_series_ticker": event.get("series_ticker", ""),
            "kalshi_title":         event.get("title", ""),
            "kalshi_subtitle":      event.get("sub_title", ""),
            "kalshi_category":      event.get("category", ""),
            "kalshi_num_markets":   len(nested_markets),
            "kalshi_market_titles": market_titles,
            "kalshi_market_titles_str": " | ".join(market_titles[:10]),
            "kalshi_rules":         market_rules[0][:500] if market_rules else "",
            "kalshi_strike_date":   event.get("strike_date", ""),
        })

    df = pd.DataFrame(rows)
    return df


# ---- EXECUTE ----
raw_kalshi_events = fetch_all_kalshi_events()
df_kalshi = parse_kalshi_events(raw_kalshi_events)

# ---- DIAGNOSTICS ----
print(f"\nDataFrame shape: {df_kalshi.shape}")
print(f"Events with titles: {(df_kalshi['kalshi_title'].str.len() > 0).sum()} / {len(df_kalshi)}")
print(f"Events with rules: {(df_kalshi['kalshi_rules'].str.len() > 0).sum()} / {len(df_kalshi)}")

print(f"\nKalshi category distribution:")
for cat, count in df_kalshi['kalshi_category'].value_counts().items():
    pct = count / len(df_kalshi) * 100
    print(f"  {cat:<25} {count:>5}  ({pct:.1f}%)")

print(f"\nSample events:")
for _, row in df_kalshi.sample(min(10, len(df_kalshi)), random_state=42).iterrows():
    print(f"  [{row['kalshi_category']}] {row['kalshi_title']}")
    if row['kalshi_subtitle']:
        print(f"    Subtitle: {row['kalshi_subtitle'][:80]}")

Fetching active Kalshi events...
  Pulled 200 events (total so far: 200)
  Pulled 200 events (total so far: 400)
  Pulled 200 events (total so far: 600)
  Pulled 200 events (total so far: 800)
  Pulled 200 events (total so far: 1000)
  Pulled 200 events (total so far: 1200)
  Pulled 200 events (total so far: 1400)
  Pulled 200 events (total so far: 1600)
  Pulled 200 events (total so far: 1800)
  Pulled 200 events (total so far: 2000)
  Pulled 200 events (total so far: 2200)
  Pulled 200 events (total so far: 2400)
  Pulled 200 events (total so far: 2600)
  Pulled 200 events (total so far: 2800)
  Pulled 200 events (total so far: 3000)
  Pulled 200 events (total so far: 3200)
  Pulled 200 events (total so far: 3400)
  Pulled 200 events (total so far: 3600)
  Pulled 200 events (total so far: 3800)
  Pulled 200 events (total so far: 4000)
  Pulled 200 events (total so far: 4200)
  Pulled 200 events (total so far: 4400)
  Pulled 200 events (total so far: 4600)
  Pulled 200 events (total s

In [ ]:
# ============================================================
# CELL 8 (REVISED): CATEGORY-GATED TF-IDF CANDIDATE MATCHING
# ============================================================
# Changes from original:
#   - Candidates now carry kalshi_event_ticker, kalshi_market_titles_str,
#     and kalshi_rules forward for Cell 9's enriched verification prompt.
# ============================================================

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

SIMILARITY_THRESHOLD = 0.30  # Deliberately low to avoid missing true matches

def normalize_title(title):
    """Light normalization to improve TF-IDF matching across platforms."""
    t = str(title).lower().strip()
    # Standardize common abbreviations
    t = t.replace("nyc", "new york city")
    t = t.replace("u.s.", "us").replace("u.s", "us")
    t = t.replace("gov.", "governor").replace("gov ", "governor ")
    # Remove punctuation that splits tokens unhelpfully
    t = re.sub(r'[?!,\.\(\)]', ' ', t)
    # Collapse whitespace
    t = re.sub(r'\s+', ' ', t).strip()
    return t


def find_candidates_in_category(df_poly_cat, df_kalshi_cat, category):
    """
    For a single category, compute TF-IDF similarity between all
    Polymarket and Kalshi titles, returning pairs above threshold.
    """
    if len(df_poly_cat) == 0 or len(df_kalshi_cat) == 0:
        return []

    # Normalize titles
    poly_titles = df_poly_cat['title'].apply(normalize_title).tolist()
    kalshi_titles = df_kalshi_cat['kalshi_title'].apply(normalize_title).tolist()

    # Also incorporate subtitle for Kalshi (adds context like dates, tickers)
    kalshi_combined = []
    for i, row in df_kalshi_cat.iterrows():
        combined = normalize_title(row['kalshi_title'])
        if pd.notna(row.get('kalshi_subtitle')) and str(row['kalshi_subtitle']).strip():
            combined += " " + normalize_title(row['kalshi_subtitle'])
        kalshi_combined.append(combined)

    # Fit TF-IDF on combined corpus
    all_texts = poly_titles + kalshi_combined
    vectorizer = TfidfVectorizer(
        ngram_range=(1, 2),   # unigrams + bigrams for phrase matching
        min_df=1,
        max_df=0.95,
        stop_words='english'
    )
    tfidf_matrix = vectorizer.fit_transform(all_texts)

    # Split back into Polymarket and Kalshi matrices
    poly_matrix = tfidf_matrix[:len(poly_titles)]
    kalshi_matrix = tfidf_matrix[len(poly_titles):]

    # Compute pairwise cosine similarity
    sim_matrix = cosine_similarity(poly_matrix, kalshi_matrix)

    # Extract candidate pairs above threshold
    candidates = []
    poly_indices = df_poly_cat.index.tolist()
    kalshi_indices = df_kalshi_cat.index.tolist()

    for i in range(sim_matrix.shape[0]):
        for j in range(sim_matrix.shape[1]):
            score = sim_matrix[i, j]
            if score >= SIMILARITY_THRESHOLD:
                candidates.append({
                    'poly_event_id':       df_poly_cat.iloc[i]['event_id'],
                    'poly_title':          df_poly_cat.iloc[i]['title'],
                    'kalshi_event_ticker': df_kalshi_cat.iloc[j]['kalshi_event_ticker'],
                    'kalshi_title':        df_kalshi_cat.iloc[j]['kalshi_title'],
                    'kalshi_subtitle':     df_kalshi_cat.iloc[j].get('kalshi_subtitle', ''),
                    # ── NEW: carry forward for Cell 9 enriched prompt ──
                    'kalshi_market_titles_str': df_kalshi_cat.iloc[j].get('kalshi_market_titles_str', ''),
                    'kalshi_rules':        df_kalshi_cat.iloc[j].get('kalshi_rules', ''),
                    # ── END NEW ──
                    'category':            category,
                    'tfidf_score':         round(score, 4),
                })

    return candidates


# ---- EXECUTE ACROSS ALL CATEGORIES ----

# Load both datasets
df_poly = pd.read_csv('polymarket_classified_final.csv')

shared_categories = set(df_poly['kalshi_category'].unique()) & set(df_kalshi['kalshi_category'].unique())

all_candidates = []
print(f"Running TF-IDF matching across {len(shared_categories)} shared categories...")
print(f"Similarity threshold: {SIMILARITY_THRESHOLD}\n")

for cat in sorted(shared_categories):
    df_p_cat = df_poly[df_poly['kalshi_category'] == cat].reset_index(drop=True)
    df_k_cat = df_kalshi[df_kalshi['kalshi_category'] == cat].reset_index(drop=True)

    candidates = find_candidates_in_category(df_p_cat, df_k_cat, cat)
    all_candidates.extend(candidates)

    print(f"  {cat:<25} Poly: {len(df_p_cat):>5} × Kalshi: {len(df_k_cat):>4} → {len(candidates):>5} candidates")

df_candidates = pd.DataFrame(all_candidates)

# ---- DIAGNOSTICS ----
print(f"\n{'='*55}")
print(f"CANDIDATE GENERATION RESULTS")
print(f"{'='*55}")
print(f"  Total candidate pairs: {len(df_candidates):,}")

if len(df_candidates) > 0:
    print(f"\n  Candidates per category:")
    for cat, count in df_candidates['category'].value_counts().items():
        print(f"    {cat:<25} {count:>5}")

    print(f"\n  TF-IDF score distribution:")
    print(f"    Mean:     {df_candidates['tfidf_score'].mean():.3f}")
    print(f"    Median:   {df_candidates['tfidf_score'].median():.3f}")
    print(f"    >=0.70:    {(df_candidates['tfidf_score'] >= 0.70).sum()}")
    print(f"    >=0.50:    {(df_candidates['tfidf_score'] >= 0.50).sum()}")
    print(f"    0.30-0.50: {((df_candidates['tfidf_score'] >= 0.30) & (df_candidates['tfidf_score'] < 0.50)).sum()}")

    # Sort by score descending
    df_candidates = df_candidates.sort_values('tfidf_score', ascending=False).reset_index(drop=True)

    # Show top 20 highest-scoring pairs
    print(f"\n  TOP 20 CANDIDATE PAIRS (highest TF-IDF):")
    print(f"  {'-'*70}")
    for _, r in df_candidates.head(20).iterrows():
        print(f"  [{r['category']}] Score: {r['tfidf_score']}")
        print(f"    Poly:   {r['poly_title']}")
        print(f"    Kalshi: {r['kalshi_title']}")
        if r['kalshi_subtitle']:
            print(f"            ({r['kalshi_subtitle']})")
        print()

    # Save checkpoint
    df_candidates.to_csv('matching_candidates.csv', index=False)
    print(f"  ✓ Saved {len(df_candidates):,} candidates to matching_candidates.csv")
else:
    print("\n  ⚠ No candidates found. Consider lowering the threshold.")


Running TF-IDF matching across 13 shared categories...
Similarity threshold: 0.3

  Climate and Weather       Poly:   167 × Kalshi:  113 →   151 candidates
  Companies                 Poly:   123 × Kalshi:   74 →    21 candidates
  Crypto                    Poly:  3647 × Kalshi:   86 →    27 candidates
  Economics                 Poly:    93 × Kalshi:  300 →   209 candidates
  Elections                 Poly:   850 × Kalshi: 1173 →  4506 candidates
  Entertainment             Poly:   143 × Kalshi:  433 →   297 candidates
  Financials                Poly:   312 × Kalshi:   59 →    16 candidates
  Mentions                  Poly:    43 × Kalshi:   36 →     9 candidates
  Politics                  Poly:   570 × Kalshi:  327 →   185 candidates
  Science and Technology    Poly:    41 × Kalshi:   65 →    10 candidates
  Social                    Poly:    57 × Kalshi:   16 →     5 candidates
  Sports                    Poly:  4829 × Kalshi: 2531 →  2733 candidates
  World                     Po

In [ ]:
# ============================================================
# CELL 8B (v3): DETERMINISTIC PRE-FILTER
# ============================================================
# v3 changes from v2:
#   - Date regex fix: (?!\d) prevents partial year extraction
#   - Auto-reject: "pure" vs non-"pure" metric mismatch
#   - Auto-reject: ordinal mismatch (1st vs 3rd round leader)
#   - Auto-reject: scope mismatch ("play in" vs "win/winner")
#   - Auto-reject: "nominal" vs non-"nominal" GDP
#   - Auto-reject: "QoQ" vs "YoY" metric mismatch
#   - Dedup: ONLY removes "More Markets" (safe noise). Kalshi
#     ticker and Poly event ID dedups moved to Cell 10 where
#     they run on verified matches, not raw candidates.
#   - Normalizer: league synonyms (Pro Football=NFL, etc.)
#   - Normalizer: strip leading numeric prefixes ("1. FC" → "FC")
#   - Normalizer: strip common club prefixes (CA, CD, SS, AD, etc.)
#   - Normalizer: "Winner" appended for bare trophy names
# ============================================================

import re

df_cand = pd.read_csv('matching_candidates.csv')
original_count = len(df_cand)

print(f"DETERMINISTIC PRE-FILTER (v3)")
print(f"{'='*55}")
print(f"  Input: {original_count:,} candidate pairs\n")

# ================================================================
# STEP 0: SAFE DEDUP ONLY
# ================================================================
# Only remove "More Markets" (always noise, always safe).
# Kalshi ticker and Poly event ID dedups are intentionally NOT
# done here — they go in Cell 10 AFTER verification.

more_mask = df_cand['poly_title'].str.contains('More Markets', na=False)
more_removed = more_mask.sum()
df_cand = df_cand[~more_mask].copy()
df_cand = df_cand.reset_index(drop=True)

print(f"  [Dedup] Removed {more_removed} 'More Markets' duplicates")
print(f"  [Dedup] {original_count:,} → {len(df_cand):,} candidates\n")


# ================================================================
# NORMALIZATION
# ================================================================

# Structural synonyms: universally equivalent in prediction markets.
SYNONYM_MAP = {
    # Competition outcome equivalences
    "champion": "winner",
    "champions": "winners",
    # Primary winner = nominee (US elections)
    "primary winner": "nominee",
    # Same framing
    "election winner": "winner",
    # Kalshi's naming convention for US leagues
    "pro football": "nfl",
    "pro basketball": "nba",
    "pro baseball": "mlb",
}

# Club/organization suffixes SAFE to strip (never carry meaning).
# NOT stripped: SC (South Carolina), AFC (NFL conference), AC, BC
SAFE_STRIP_SUFFIXES = [
    r'\bfc\b', r'\bcf\b', r'\bssc\b',
    r'\bud\b', r'\bfk\b', r'\bsk\b',
    r'\bkk\b', r'\brc\b',
    # Founding years in club names
    r'\b1909\b', r'\b1912\b', r'\b1846\b', r'\b1899\b', r'\b1901\b',
    r'\b1929\b', r'\b1963\b',
]

# Leading numeric prefixes in club names ("1. FC Köln" → "FC Köln")
LEADING_NUM_PREFIX = r'^\d+\.\s*'

# Common club name prefixes that are noise (CA = Club Atlético, etc.)
# Only stripped as leading tokens, not mid-name
CLUB_PREFIXES = [
    r'^ca\b', r'^cd\b', r'^ss\b', r'^ad\b', r'^us\b',
    r'^aa\b', r'^as\b', r'^cs\b', r'^sdp?\b',
]

# Filler words. EXCLUDED: "in" (Indiana), "or" (Oregon)
FILLER_WORDS = {
    'will', 'the', 'be', 'of', 'for', 'a', 'an', 'to', 'on',
    'at', 'by', 'and', 'is', 'vs', 'vs.', 'what', 'who',
    'which', 'how', 'next', 'this', 'that',
}

# ALL CAPS tokens 2-4 chars are protected UNLESS they're known noise
NOISE_CAPS = {'FC', 'CF', 'SSC', 'UD', 'CD', 'FK', 'SK', 'KK', 'RC'}


def extract_protected_tokens(title):
    """Find ALL CAPS 2-4 char tokens that are likely meaningful codes."""
    protected = set()
    raw_tokens = re.findall(r'[A-Za-z0-9]+', str(title))
    for tok in raw_tokens:
        if tok.isupper() and 2 <= len(tok) <= 4 and tok not in NOISE_CAPS:
            protected.add(tok.lower())
    return protected


def extract_years(title):
    return set(re.findall(r'\b(20[2-3]\d)\b', str(title)))


def normalize_for_matching(title, protected_tokens, strip_years):
    """Aggressive normalization for deterministic matching."""
    t = str(title).lower().strip()

    # Remove punctuation
    t = re.sub(r"[^\w\s]", " ", t)

    # Strip leading numeric prefix ("1 fc köln" → "fc köln")
    t = re.sub(LEADING_NUM_PREFIX, '', t)

    # Apply synonym map (longer phrases first)
    for old, new in sorted(SYNONYM_MAP.items(), key=lambda x: -len(x[0])):
        t = t.replace(old, new)

    # Strip SAFE club suffixes only
    for pattern in SAFE_STRIP_SUFFIXES:
        t = re.sub(pattern, ' ', t)

    # Strip club prefixes (only at start of normalized title)
    for pattern in CLUB_PREFIXES:
        t = re.sub(pattern, '', t).strip()

    # Tokenize
    tokens = t.split()

    # Remove filler — but NOT protected tokens
    tokens = [tok for tok in tokens
              if tok in protected_tokens
              or (tok not in FILLER_WORDS and tok.strip())]

    # Handle years: only strip if both sides share the same years
    if strip_years:
        tokens = [tok for tok in tokens
                  if tok in protected_tokens
                  or not re.match(r'^20[2-3]\d$', tok)]

    return set(tokens)


def jaccard(set_a, set_b):
    if not set_a and not set_b:
        return 1.0
    if not set_a or not set_b:
        return 0.0
    return len(set_a & set_b) / len(set_a | set_b)


# ================================================================
# AUTO-REJECT RULES
# ================================================================

def extract_group_letter(title):
    m = re.search(r'group\s+([a-z])\b', title.lower())
    return m.group(1) if m else None


def extract_specific_date(title):
    """Extract specific date. (?!\d) prevents matching "20" from "2026"."""
    m = re.search(
        r'(jan|feb|mar|apr|may|jun|jul|aug|sep|oct|nov|dec)\w*\.?\s+(\d{1,2})(?!\d)',
        title.lower()
    )
    if m:
        return f"{m.group(1)[:3]}_{m.group(2)}"
    return None


def extract_threshold(title):
    m = re.search(r'\$[\d,.]+[kmbt]?', title.lower())
    if m:
        return m.group(0)
    m = re.search(r'[\d.]+%', title.lower())
    if m:
        return m.group(0)
    return None


def extract_ordinal(title):
    """Extract ordinal positions: 1st, 2nd, 3rd, first, second, third, etc."""
    t = title.lower()
    # Map word ordinals to numbers
    word_map = {'first': 1, 'second': 2, 'third': 3, 'fourth': 4, 'fifth': 5}
    for word, num in word_map.items():
        if re.search(r'\b' + word + r'\b', t):
            return num
    # Numeric ordinals
    m = re.search(r'\b(\d+)(?:st|nd|rd|th)\b', t)
    if m:
        return int(m.group(1))
    return None


def has_keyword(title, keywords):
    """Check if title contains any of the keywords as whole words."""
    t = title.lower()
    for kw in keywords:
        if re.search(r'\b' + re.escape(kw) + r'\b', t):
            return True
    return False


def should_auto_reject(poly_title, kalshi_title):
    """
    Check for structural mismatches that guarantee NO_MATCH.
    Returns (True, reason) or (False, None).
    """
    p_low = poly_title.lower()
    k_low = kalshi_title.lower()

    # Rule 1: Different group letters
    p_group = extract_group_letter(poly_title)
    k_group = extract_group_letter(kalshi_title)
    if p_group and k_group and p_group != k_group:
        return True, f"Different groups: {p_group} vs {k_group}"

    # Rule 2: Different specific dates
    p_date = extract_specific_date(poly_title)
    k_date = extract_specific_date(kalshi_title)
    if p_date and k_date and p_date != k_date:
        return True, f"Different dates: {p_date} vs {k_date}"

    # Rule 3: Different numeric thresholds
    p_thresh = extract_threshold(poly_title)
    k_thresh = extract_threshold(kalshi_title)
    if p_thresh and k_thresh and p_thresh != k_thresh:
        return True, f"Different thresholds: {p_thresh} vs {k_thresh}"

    # Rule 4: "top N" vs "winner" (subset vs single outcome)
    p_top = bool(re.search(r'top\s+\d+', p_low))
    k_top = bool(re.search(r'top\s+\d+', k_low))
    p_win = has_keyword(poly_title, ['winner', 'champion'])
    k_win = has_keyword(kalshi_title, ['winner', 'champion'])
    if (p_top and k_win and not k_top):
        return True, "Subset mismatch: top-N vs winner"
    if (k_top and p_win and not p_top):
        return True, "Subset mismatch: winner vs top-N"

    # Rule 5: Different years (gap ≥ 2 only)
    p_years = extract_years(poly_title)
    k_years = extract_years(kalshi_title)
    if p_years and k_years and p_years.isdisjoint(k_years):
        min_gap = min(abs(int(py) - int(ky)) for py in p_years for ky in k_years)
        if min_gap >= 2:
            return True, f"Different years: {p_years} vs {k_years}"

    # Rule 6: "pure" metric mismatch
    # One side says "pure" (sales only) and the other doesn't.
    # Different measurement = different resolution.
    p_pure = has_keyword(poly_title, ['pure'])
    k_pure = has_keyword(kalshi_title, ['pure'])
    if p_pure != k_pure:
        # Only reject if both are about sales/activity
        if has_keyword(poly_title, ['sales', 'album', 'activity', 'first week']) or \
           has_keyword(kalshi_title, ['sales', 'album', 'activity', 'first week']):
            return True, "Metric mismatch: pure vs combined/activity"

    # Rule 7: "nominal" GDP mismatch
    # "nominal GDP" ≠ "GDP" (real/unspecified)
    p_nominal = has_keyword(poly_title, ['nominal'])
    k_nominal = has_keyword(kalshi_title, ['nominal'])
    if p_nominal != k_nominal:
        if has_keyword(poly_title, ['gdp']) or has_keyword(kalshi_title, ['gdp']):
            return True, "Metric mismatch: nominal GDP vs real GDP"

    # Rule 8: QoQ vs YoY mismatch
    p_qoq = has_keyword(poly_title, ['qoq'])
    k_qoq = has_keyword(kalshi_title, ['qoq'])
    p_yoy = has_keyword(poly_title, ['yoy'])
    k_yoy = has_keyword(kalshi_title, ['yoy'])
    if (p_qoq and k_yoy) or (p_yoy and k_qoq):
        return True, "Metric mismatch: QoQ vs YoY"

    # Rule 9: Ordinal mismatch (1st round leader ≠ 3rd round leader)
    # Only apply when both titles have ordinals AND share context
    # (e.g. both about "round leader" or "overall pick")
    p_ord = extract_ordinal(poly_title)
    k_ord = extract_ordinal(kalshi_title)
    if p_ord is not None and k_ord is not None and p_ord != k_ord:
        # Check shared context to avoid false triggers
        shared_context = ['round', 'leader', 'pick', 'place', 'seed',
                         'half', 'quarter', 'period', 'inning']
        if any(has_keyword(poly_title, [ctx]) and has_keyword(kalshi_title, [ctx])
               for ctx in shared_context):
            return True, f"Ordinal mismatch: {p_ord} vs {k_ord}"

    # Rule 10: "play in" vs "win" scope mismatch
    # "Will X play in the World Cup?" ≠ "World Cup Winner"
    p_play = has_keyword(poly_title, ['play in', 'participate', 'appear'])
    k_play = has_keyword(kalshi_title, ['play in', 'participate', 'appear'])
    if (p_play and k_win and not p_win) or (k_play and p_win and not k_win):
        return True, "Scope mismatch: play/participate vs win"

    return False, None


# ================================================================
# CLASSIFY EACH CANDIDATE
# ================================================================

AUTO_MATCH_THRESHOLD = 0.85

auto_match = []
auto_reject = []
needs_llm = []

for _, row in df_cand.iterrows():
    poly_title = str(row['poly_title'])
    kalshi_title = str(row['kalshi_title'])

    # Check auto-reject first
    reject, reason = should_auto_reject(poly_title, kalshi_title)
    if reject:
        row_dict = row.to_dict()
        row_dict['verdict'] = 'NO_MATCH'
        row_dict['reason'] = f'Auto-reject: {reason}'
        auto_reject.append(row_dict)
        continue

    # Extract protected tokens from ORIGINAL casing
    p_protected = extract_protected_tokens(poly_title)
    k_protected = extract_protected_tokens(kalshi_title)
    all_protected = p_protected | k_protected

    # Year handling
    p_years = extract_years(poly_title)
    k_years = extract_years(kalshi_title)
    strip_years = (p_years == k_years)

    # Normalize and score
    p_tokens = normalize_for_matching(poly_title, all_protected, strip_years)
    k_tokens = normalize_for_matching(kalshi_title, all_protected, strip_years)
    sim = jaccard(p_tokens, k_tokens)

    if sim >= AUTO_MATCH_THRESHOLD:
        row_dict = row.to_dict()
        row_dict['verdict'] = 'MATCH'
        row_dict['reason'] = f'Auto-match: Jaccard={sim:.3f} after normalization'
        auto_match.append(row_dict)
    else:
        needs_llm.append(row.to_dict())


# ================================================================
# REPORT
# ================================================================

print(f"  Auto-MATCH:  {len(auto_match):>5}  (Jaccard ≥ {AUTO_MATCH_THRESHOLD} after normalization)")
print(f"  Auto-REJECT: {len(auto_reject):>5}  (structural mismatch)")
print(f"  Needs LLM:   {len(needs_llm):>5}  (ambiguous → Cell 9)")
print(f"  Total:       {len(auto_match) + len(auto_reject) + len(needs_llm):>5}")

if auto_match:
    df_am = pd.DataFrame(auto_match)
    print(f"\n  Auto-MATCH by category:")
    for cat, count in df_am['category'].value_counts().items():
        print(f"    {cat:<30} {count:>5}")

    print(f"\n  Auto-MATCH samples (15 random):")
    print(f"  {'-'*60}")
    for _, r in df_am.sample(min(15, len(df_am)), random_state=42).iterrows():
        print(f"  [{r['category']}] {r['reason']}")
        print(f"    Poly:   {r['poly_title']}")
        print(f"    Kalshi: {r['kalshi_title']}")
        print()

if auto_reject:
    df_ar = pd.DataFrame(auto_reject)
    print(f"\n  Auto-REJECT by reason pattern:")
    reason_prefixes = ['Different groups', 'Different dates', 'Different thresholds',
                       'Subset mismatch', 'Different years', 'Metric mismatch',
                       'Ordinal mismatch', 'Scope mismatch']
    for rp in reason_prefixes:
        count = df_ar['reason'].str.contains(rp).sum()
        if count > 0:
            print(f"    {rp:<35} {count:>5}")

    print(f"\n  Auto-REJECT samples (10 random):")
    print(f"  {'-'*60}")
    for _, r in df_ar.sample(min(10, len(df_ar)), random_state=42).iterrows():
        print(f"  [{r['category']}] {r['reason']}")
        print(f"    Poly:   {r['poly_title']}")
        print(f"    Kalshi: {r['kalshi_title']}")
        print()

df_llm_bound = pd.DataFrame(needs_llm)
if len(df_llm_bound) > 0:
    print(f"\n  LLM-bound TF-IDF distribution:")
    print(f"    ≥0.70:     {(df_llm_bound['tfidf_score'] >= 0.70).sum()}")
    print(f"    0.50-0.70: {((df_llm_bound['tfidf_score'] >= 0.50) & (df_llm_bound['tfidf_score'] < 0.70)).sum()}")
    print(f"    0.30-0.50: {((df_llm_bound['tfidf_score'] >= 0.30) & (df_llm_bound['tfidf_score'] < 0.50)).sum()}")

# Save outputs
df_auto_matches = pd.DataFrame(auto_match) if auto_match else pd.DataFrame()
df_needs_llm_verification = pd.DataFrame(needs_llm) if needs_llm else pd.DataFrame()

if len(df_auto_matches) > 0:
    df_auto_matches.to_csv('auto_matches.csv', index=False)

if len(df_needs_llm_verification) > 0:
    df_needs_llm_verification.to_csv('llm_candidates.csv', index=False)

print(f"\n{'='*55}")
print(f"  ✓ {len(auto_match)} auto-matches saved to auto_matches.csv")
print(f"  ✓ {len(needs_llm)} LLM candidates saved to llm_candidates.csv")
est_batches = (len(needs_llm) + 9) // 10
est_cost = est_batches * 0.04
print(f"  ✓ Estimated Cell 9 cost: ~${est_cost:.2f} ({est_batches} batches of 10)")

DETERMINISTIC PRE-FILTER (v3)
  Input: 8,169 candidate pairs

  [Dedup] Removed 221 'More Markets' duplicates
  [Dedup] 8,169 → 7,948 candidates



<>:167: SyntaxWarning: invalid escape sequence '\d'
<>:167: SyntaxWarning: invalid escape sequence '\d'
/tmp/ipykernel_4748/2109413417.py:167: SyntaxWarning: invalid escape sequence '\d'
  """Extract specific date. (?!\d) prevents matching "20" from "2026"."""


  Auto-MATCH:    785  (Jaccard ≥ 0.85 after normalization)
  Auto-REJECT:   394  (structural mismatch)
  Needs LLM:    6769  (ambiguous → Cell 9)
  Total:        7948

  Auto-MATCH by category:
    Elections                        639
    Sports                           117
    Entertainment                     12
    Economics                          6
    Politics                           4
    Financials                         2
    Mentions                           2
    Companies                          2
    Crypto                             1

  Auto-MATCH samples (15 random):
  ------------------------------------------------------------
  [Sports] Auto-match: Jaccard=1.000 after normalization
    Poly:   Toronto FC vs. Austin FC
    Kalshi: Toronto vs Austin

  [Elections] Auto-match: Jaccard=1.000 after normalization
    Poly:   CA-36 House Election Winner
    Kalshi: CA-36 House winner?

  [Elections] Auto-match: Jaccard=1.000 after normalization
    Poly:   RI-01 Hou

In [ ]:
# ============================================================
# CELL 9 (REVISED): LLM BATCH VERIFICATION OF CANDIDATE PAIRS
# ============================================================
# Changes from previous revision:
#   1. Reads llm_candidates.csv (from Cell 8B) instead of
#      matching_candidates.csv — only sees ambiguous pairs
#   2. Batch size increased from 5 to 10 (~50% cost reduction)
#   3. Prompt rebalanced: still conservative, but with explicit
#      MATCH examples for structurally equivalent patterns to
#      reduce false negatives
#   4. Merges LLM results with auto_matches.csv at the end
#   5. max_tokens increased to 4000 for larger batches
# ============================================================

BATCH_SIZE = 10

MATCH_SYSTEM_PROMPT = """You are a prediction market verification engine. You will receive pairs of events from Polymarket and Kalshi. Your job is to determine whether each pair resolves based on the EXACT SAME real-world outcome.

This verification gates a live arbitrage system. Errors in EITHER direction are costly:
- False MATCH → the system trades two contracts that resolve differently → GUARANTEED LOSS
- False NO_MATCH → the system misses a profitable arbitrage opportunity → LOST REVENUE

Your goal is ACCURACY, not caution. Output MATCH when the events resolve on the same outcome. Output NO_MATCH when you have identified a specific, concrete difference in resolution criteria.

INSTRUCTIONS — follow these steps for each pair:

Step 1: RESOLUTION CRITERIA. For each side independently, state:
  - What specific real-world outcome determines resolution?
  - What data source or authority is used (if stated)?
  - What is the exact timeframe or deadline?
  Do NOT assume they are the same. Read each side's title, sub-markets, and rules carefully.

Step 2: COMPARE. Are the resolution criteria identical? Check for:

  THESE ARE MATCHES (same resolution, different wording):
  - Different wording for the same question ("Election Winner" = "winner?", "Champion" = "Winner")
  - "by...?" date bracket events match "When will...?" events (same underlying question — will X happen, with time-bucketed sub-markets)
  - Equivalent timeframes ("before 2027" = "by December 31, 2026" = "in 2026")
  - Team name formatting differences ("FC Barcelona" = "Barcelona", "1. FC Köln" = "FC Köln" = "Köln")
  - Player name formatting ("Alexander Zverev" = "Zverev")

  THESE ARE NOT MATCHES (different resolution):
  - Different data sources ("Spotify streams" ≠ "IFPI global sales")
  - Different scope ("Last place / 20th" ≠ "Relegated / bottom 3")
  - Different thresholds ("$100k" ≠ "$150k")
  - Different dates ("March 25" ≠ "March 27")
  - Different election cycles (check ticker suffixes: "-28" = 2028, "-26" = 2026)
  - Different groups/divisions ("Group A" ≠ "Group D")
  - One is a subset of the other ("Top 3" ≠ "Winner")
  - Different metrics ("GDP" ≠ "nominal GDP", "QoQ" ≠ "YoY")

Step 3: VERDICT. Output MATCH if the resolution criteria are the same. Output NO_MATCH only if you identified a specific difference in Step 2. Do not reject pairs based on vague uncertainty — state the concrete difference or match.

For each pair, respond with a JSON object containing:
  - "pair": the pair number
  - "poly_resolution": what the Polymarket side resolves on (1 sentence)
  - "kalshi_resolution": what the Kalshi side resolves on (1 sentence)
  - "verdict": "MATCH" or "NO_MATCH"
  - "reason": one sentence explaining the specific match or the specific difference

Return your answers as a JSON array:
[{"pair": 1, "poly_resolution": "...", "kalshi_resolution": "...", "verdict": "MATCH", "reason": "..."}, ...]"""


def build_batch_prompt(batch):
    """Build a prompt for a batch of candidate pairs with enriched context."""
    lines = []
    for i, (_, row) in enumerate(batch.iterrows(), 1):
        kalshi_display = row['kalshi_title']
        if row.get('kalshi_subtitle') and pd.notna(row['kalshi_subtitle']) and str(row['kalshi_subtitle']).strip():
            kalshi_display += f" ({row['kalshi_subtitle']})"

        # Build enriched context block
        context_parts = [
            f"  Polymarket: {row['poly_title']}",
            f"  Kalshi: {kalshi_display}",
            f"  Kalshi ticker: {row['kalshi_event_ticker']}",
            f"  Category: {row['category']}",
        ]

        # Add sub-market titles if available (reveals resolution structure)
        mkt_titles = row.get('kalshi_market_titles_str', '')
        if pd.notna(mkt_titles) and str(mkt_titles).strip():
            context_parts.append(f"  Kalshi sub-markets: {str(mkt_titles)[:300]}")

        # Add resolution rules if available
        rules = row.get('kalshi_rules', '')
        if pd.notna(rules) and str(rules).strip():
            context_parts.append(f"  Kalshi rules: {str(rules)[:400]}")

        lines.append(f"PAIR {i}:\n" + "\n".join(context_parts))

    return "\n\n".join(lines) + "\n\nReturn ONLY the JSON array."


def verify_batch(batch, max_retries=3):
    """Send a batch of pairs to Claude for verification."""
    prompt = build_batch_prompt(batch)

    for attempt in range(max_retries):
        try:
            response = client.messages.create(
                model="claude-sonnet-4-20250514",
                max_tokens=4000,
                system=MATCH_SYSTEM_PROMPT,
                messages=[{"role": "user", "content": prompt}]
            )

            raw_text = response.content[0].text.strip()
            results = extract_json_array(raw_text)

            if results is None or len(results) == 0:
                if attempt < max_retries - 1:
                    time.sleep(2)
                    continue
                return [{"pair": i+1, "verdict": "ERROR", "reason": "Parse failed"} for i in range(len(batch))]

            # Safety net: if LLM identified different resolution criteria
            # but still said MATCH, override to NO_MATCH
            for r in results:
                if r.get("verdict", "").upper() == "MATCH":
                    poly_res = r.get("poly_resolution", "").lower()
                    kalshi_res = r.get("kalshi_resolution", "").lower()
                    if poly_res and kalshi_res and poly_res != kalshi_res:
                        p_tokens = set(poly_res.split())
                        k_tokens = set(kalshi_res.split())
                        overlap = len(p_tokens & k_tokens) / max(len(p_tokens | k_tokens), 1)
                        if overlap < 0.5:
                            r["verdict"] = "NO_MATCH"
                            r["reason"] = f"Override: resolution descriptions diverge. Poly='{poly_res[:80]}' vs Kalshi='{kalshi_res[:80]}'"

            return results

        except Exception as e:
            if '429' in str(e) or 'rate' in str(e).lower():
                time.sleep(min(2 ** (attempt + 2), 60))
                continue
            if attempt < max_retries - 1:
                time.sleep(2)
                continue
            return [{"pair": i+1, "verdict": "ERROR", "reason": str(e)[:100]} for i in range(len(batch))]


def extract_json_array(text):
    """Extract a JSON array from model output."""
    text = text.strip()

    # Direct parse
    try:
        result = json.loads(text)
        if isinstance(result, list):
            return result
    except json.JSONDecodeError:
        pass

    # Strip markdown
    if '```' in text:
        cleaned = re.sub(r'```(?:json)?\s*', '', text)
        cleaned = re.sub(r'```', '', cleaned).strip()
        try:
            result = json.loads(cleaned)
            if isinstance(result, list):
                return result
        except json.JSONDecodeError:
            pass

    # Find [ ... ] block
    match = re.search(r'\[.*\]', text, re.DOTALL)
    if match:
        try:
            result = json.loads(match.group())
            if isinstance(result, list):
                return result
        except json.JSONDecodeError:
            pass

    return None


# ---- EXECUTE ----

# Load LLM-bound candidates from Cell 8B (NOT matching_candidates.csv)
df_cand = pd.read_csv('llm_candidates.csv')

# Sort by score descending (verify best matches first)
df_cand = df_cand.sort_values('tfidf_score', ascending=False).reset_index(drop=True)

total_batches = (len(df_cand) + BATCH_SIZE - 1) // BATCH_SIZE
verdicts = []
errors = 0
matches_found = 0

print(f"Starting LLM verification of {len(df_cand):,} candidate pairs...")
print(f"Batch size: {BATCH_SIZE} | Total batches: {total_batches}")
print(f"Estimated time: {total_batches * 1.5 / 60:.0f}-{total_batches * 3.0 / 60:.0f} minutes")
print(f"Estimated cost: ~${total_batches * 0.04:.2f}\n")

for batch_idx in tqdm(range(total_batches), desc="Verifying"):
    start = batch_idx * BATCH_SIZE
    end = min(start + BATCH_SIZE, len(df_cand))
    batch = df_cand.iloc[start:end]

    results = verify_batch(batch)

    # Map results back to candidate rows
    for i, (_, row) in enumerate(batch.iterrows()):
        if i < len(results):
            r = results[i]
            verdict = r.get("verdict", "ERROR")
            reason = r.get("reason", "")
        else:
            verdict = "ERROR"
            reason = "Missing from batch response"

        verdicts.append({
            'poly_event_id':       row['poly_event_id'],
            'poly_title':          row['poly_title'],
            'kalshi_event_ticker': row['kalshi_event_ticker'],
            'kalshi_title':        row['kalshi_title'],
            'kalshi_subtitle':     row.get('kalshi_subtitle', ''),
            'category':            row['category'],
            'tfidf_score':         row['tfidf_score'],
            'verdict':             verdict.upper().strip(),
            'reason':              reason,
        })

        if verdict.upper().strip() == "MATCH":
            matches_found += 1
        elif verdict.upper().strip() == "ERROR":
            errors += 1

    time.sleep(0.3)

    # Checkpoint
    if (batch_idx + 1) % 50 == 0:
        temp_df = pd.DataFrame(verdicts)
        temp_df.to_csv('verification_checkpoint.csv', index=False)
        print(f"\n  💾 Checkpoint: batch {batch_idx+1}/{total_batches} | {matches_found} matches | {errors} errors")

# ---- MERGE: LLM MATCHES + AUTO-MATCHES FROM CELL 8B ----

df_verdicts = pd.DataFrame(verdicts)
df_llm_matches = df_verdicts[df_verdicts['verdict'] == 'MATCH'].copy()
df_no_match = df_verdicts[df_verdicts['verdict'] == 'NO_MATCH'].copy()
df_errors = df_verdicts[df_verdicts['verdict'] == 'ERROR'].copy()

# Load auto-matches from Cell 8B
import os
if os.path.exists('auto_matches.csv'):
    df_auto = pd.read_csv('auto_matches.csv')
    auto_count = len(df_auto)
else:
    df_auto = pd.DataFrame()
    auto_count = 0

# Combine auto-matches + LLM matches
output_cols = ['poly_event_id', 'poly_title', 'kalshi_event_ticker',
               'kalshi_title', 'kalshi_subtitle', 'category',
               'tfidf_score', 'verdict', 'reason']

# Ensure both DataFrames have the required columns
for col in output_cols:
    if col not in df_llm_matches.columns:
        df_llm_matches[col] = ''
    if len(df_auto) > 0 and col not in df_auto.columns:
        df_auto[col] = ''

df_all_matches = pd.concat(
    [df_auto[output_cols], df_llm_matches[output_cols]] if len(df_auto) > 0
    else [df_llm_matches[output_cols]],
    ignore_index=True
)

# ---- FINAL RESULTS ----

print(f"\n{'='*55}")
print(f"VERIFICATION RESULTS")
print(f"{'='*55}")
print(f"  Auto-matched (Cell 8B):  {auto_count:,}")
print(f"  LLM verified:            {len(df_verdicts):,}")
print(f"    LLM MATCH:             {len(df_llm_matches):,}")
print(f"    LLM NO_MATCH:          {len(df_no_match):,}")
print(f"    LLM ERROR:             {len(df_errors):,}")
print(f"  ────────────────────────────")
print(f"  TOTAL CONFIRMED MATCHES: {len(df_all_matches):,}")

if len(df_all_matches) > 0:
    print(f"\n  Matches by category:")
    for cat, count in df_all_matches['category'].value_counts().items():
        print(f"    {cat:<30} {count:>4}")

    print(f"\n  Match source breakdown:")
    auto_in_final = df_all_matches['reason'].str.startswith('Auto-match').sum()
    llm_in_final = len(df_all_matches) - auto_in_final
    print(f"    Auto-matched:  {auto_in_final:>5}")
    print(f"    LLM-verified:  {llm_in_final:>5}")

    print(f"\n  TF-IDF score of all matches:")
    print(f"    Mean:   {df_all_matches['tfidf_score'].mean():.3f}")
    print(f"    Min:    {df_all_matches['tfidf_score'].min():.3f}")
    print(f"    Max:    {df_all_matches['tfidf_score'].max():.3f}")

    print(f"\n  ALL CONFIRMED MATCHES:")
    print(f"  {'-'*70}")
    for _, r in df_all_matches.iterrows():
        print(f"  [{r['category']}] TF-IDF: {r['tfidf_score']}")
        print(f"    Poly:   {r['poly_title']}")
        print(f"    Kalshi: {r['kalshi_title']}")
        if r['kalshi_subtitle'] and pd.notna(r['kalshi_subtitle']):
            print(f"            ({r['kalshi_subtitle']})")
        print(f"    Reason: {r['reason']}")
        print()

    df_all_matches.to_csv('confirmed_matches.csv', index=False)
    print(f"  ✓ Saved {len(df_all_matches)} confirmed matches to confirmed_matches.csv")

# Save full verification log
df_verdicts.to_csv('verification_checkpoint.csv', index=False)

# Cost summary
est_cost = total_batches * 0.04
print(f"\n  Estimated LLM cost: ~${est_cost:.2f}")
print(f"  Savings vs full verification: ~${(len(df_cand) / 5 * 0.04) - est_cost:.2f} (vs sending all to LLM)")

Starting LLM verification of 6,769 candidate pairs...
Batch size: 10 | Total batches: 677
Estimated time: 17-34 minutes
Estimated cost: ~$27.08



Verifying:   0%|          | 0/677 [00:00<?, ?it/s]

Streaming output truncated to the last 5000 lines.
    Kalshi: NY-15 Democratic nominee?
            (In 2026)
    Reason: Auto-match: Jaccard=1.000 after normalization

  [Elections] TF-IDF: 0.7854
    Poly:   NY-17 Democratic Primary Winner
    Kalshi: NY-17 Democratic nominee?
            (In 2026)
    Reason: Auto-match: Jaccard=1.000 after normalization

  [Elections] TF-IDF: 0.7846
    Poly:   MI-11 Democratic Primary Winner
    Kalshi: MI-11 Democratic nominee?
            (In 2026)
    Reason: Auto-match: Jaccard=1.000 after normalization

  [Elections] TF-IDF: 0.7835
    Poly:   GA-09 Republican Primary Winner
    Kalshi: GA-09 Republican nominee?
            (GA-09 (R))
    Reason: Auto-match: Jaccard=1.000 after normalization

  [Politics] TF-IDF: 0.7832
    Poly:   Will the US reopen its embassy in Iran in 2026?
    Kalshi: Will the US reopen its embassy in Iran?
            (Before 2027)
    Reason: Auto-match: Jaccard=0.857 after normalization

  [Elections] TF-IDF: 0.782

In [ ]:
# ============================================================
# CELL 10 (v3): POST-VERIFICATION DEDUP & CLEANUP
# ============================================================
# Runs AFTER Cell 9 on confirmed_matches.csv (verified pairs).
# At this point every row is a verified match — dedup only
# removes true duplicates, not true matches.
#
# 1. Dedup on Kalshi ticker (keep highest TF-IDF)
# 2. Dedup on Poly event ID (keep highest TF-IDF)
#
# Because these run on VERIFIED matches, the "NHL Atlantic"
# vs "NBA Atlantic" problem from v2 can't happen — the LLM
# already confirmed which one is correct.
# ============================================================

df_matches = pd.read_csv('confirmed_matches.csv')
original_count = len(df_matches)

print(f"POST-VERIFICATION DEDUP (v3)")
print(f"{'='*55}")
print(f"  Input: {original_count} confirmed matches\n")

# ── STEP 1: Dedup on Kalshi ticker ──
# Same Kalshi event verified against multiple Poly events.
# Keep highest TF-IDF (strongest textual match).
df_matches = df_matches.sort_values('tfidf_score', ascending=False)
before = len(df_matches)
df_matches = df_matches.drop_duplicates(subset='kalshi_event_ticker', keep='first')
ticker_removed = before - len(df_matches)
print(f"  [1] Removed {ticker_removed} duplicate Kalshi tickers")

# ── STEP 2: Dedup on Poly event ID ──
# Same Poly event verified against multiple Kalshi events.
# Keep highest TF-IDF.
before = len(df_matches)
df_matches = df_matches.drop_duplicates(subset='poly_event_id', keep='first')
poly_removed = before - len(df_matches)
print(f"  [2] Removed {poly_removed} duplicate Poly event IDs")

# ── SAVE ──
df_matches = df_matches.reset_index(drop=True)
df_matches.to_csv('confirmed_matches.csv', index=False)

print(f"\n  Total removed: {ticker_removed + poly_removed}")
print(f"  Final count:   {len(df_matches)} confirmed matches")

print(f"\n  Category breakdown:")
for cat, count in df_matches['category'].value_counts().items():
    print(f"    {cat:<30} {count:>5}")

print(f"\n  TF-IDF score stats:")
print(f"    Mean:   {df_matches['tfidf_score'].mean():.3f}")
print(f"    Min:    {df_matches['tfidf_score'].min():.3f}")
print(f"    Median: {df_matches['tfidf_score'].median():.3f}")

print(f"\n{'='*55}")
print(f"✓ Cleaned confirmed_matches.csv saved ({len(df_matches)} matches)")

POST-VERIFICATION DEDUP (v3)
  Input: 1297 confirmed matches

  [1] Removed 55 duplicate Kalshi tickers
  [2] Removed 23 duplicate Poly event IDs

  Total removed: 78
  Final count:   1219 confirmed matches

  Category breakdown:
    Elections                        670
    Sports                           452
    Entertainment                     36
    Politics                          20
    Economics                         18
    Climate and Weather               11
    Companies                          4
    Mentions                           3
    Financials                         2
    Science and Technology             2
    Crypto                             1

  TF-IDF score stats:
    Mean:   0.674
    Min:    0.300
    Median: 0.732

✓ Cleaned confirmed_matches.csv saved (1219 matches)
